# Fashion Recommender System with KNN

## Building a simple, interpretable fashion recommender system using K-Nearest Neighbors (KNN).

- creates a small synthetic fashion dataset
- represents users with preference features
- recommends items based on similar users
- explains recommendations with nearest-neighbor reasoning

### Tutorial from the Explainable AI Class of the German University of Digital

## 1. Why KNN for recommendations?

KNN is useful in an explainability context because it makes predictions using similar examples.

Instead of a hidden black-box explanation, we can say:

> This item was recommended because users with similar style preferences liked it.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

np.random.seed(42)

## 2. Create a synthetic user dataset

Each user has a profile with preferences such as:
- casual style
- elegant style
- sporty style
- neutral colors
- bright colors
- budget sensitivity

In [ ]:
n_users = 20

users = pd.DataFrame({
    "user_id": [f"U{i:02d}" for i in range(1, n_users + 1)],
    "casual": np.random.randint(1, 6, n_users),
    "elegant": np.random.randint(1, 6, n_users),
    "sporty": np.random.randint(1, 6, n_users),
    "neutral_colors": np.random.randint(1, 6, n_users),
    "bright_colors": np.random.randint(1, 6, n_users),
    "budget_sensitive": np.random.randint(1, 6, n_users),
})

users.head()

,user_id,casual,elegant,sporty,neutral_colors,bright_colors,budget_sensitive
0,U01,4,2,4,2,5,4
1,U02,5,5,1,4,1,1
2,U03,3,4,4,5,5,4
3,U04,5,1,2,2,5,2
4,U05,5,1,2,2,1,1


## 3. Create a small fashion catalog

In [ ]:
items = pd.DataFrame({
    "item_id": ["I01", "I02", "I03", "I04", "I05", "I06", "I07", "I08"],
    "item_name": [
        "Beige Trench Coat",
        "White Sneakers",
        "Black Evening Dress",
        "Blue Denim Jacket",
        "Red Sports Leggings",
        "Cream Knit Sweater",
        "Gold Heels",
        "Minimalist Tote Bag",
    ],
    "style_main": [
        "elegant",
        "casual",
        "elegant",
        "casual",
        "sporty",
        "casual",
        "elegant",
        "elegant",
    ],
    "color_type": [
        "neutral",
        "neutral",
        "neutral",
        "bright",
        "bright",
        "neutral",
        "bright",
        "neutral",
    ],
    "price_level": [
        "mid",
        "mid",
        "high",
        "mid",
        "low",
        "mid",
        "high",
        "mid",
    ]
})

items

,item_id,item_name,style_main,color_type,price_level
0,I01,Beige Trench Coat,elegant,neutral,mid
1,I02,White Sneakers,casual,neutral,mid
2,I03,Black Evening Dress,elegant,neutral,high
3,I04,Blue Denim Jacket,casual,bright,mid
4,I05,Red Sports Leggings,sporty,bright,low
5,I06,Cream Knit Sweater,casual,neutral,mid
6,I07,Gold Heels,elegant,bright,high
7,I08,Minimalist Tote Bag,elegant,neutral,mid


## 4. Simulate user-item interactions

In [ ]:
def likes_item(user_row, item_row):
    score = 0

    if item_row["style_main"] == "casual":
        score += user_row["casual"]
    elif item_row["style_main"] == "elegant":
        score += user_row["elegant"]
    elif item_row["style_main"] == "sporty":
        score += user_row["sporty"]

    if item_row["color_type"] == "neutral":
        score += user_row["neutral_colors"]
    else:
        score += user_row["bright_colors"]

    if item_row["price_level"] == "high":
        score += max(0, 6 - user_row["budget_sensitive"])
    elif item_row["price_level"] == "low":
        score += user_row["budget_sensitive"]
    else:
        score += 3

    return 1 if score >= 8 else 0

interaction_rows = []
for _, user in users.iterrows():
    row = {"user_id": user["user_id"]}
    for _, item in items.iterrows():
        row[item["item_id"]] = likes_item(user, item)
    interaction_rows.append(row)

interactions = pd.DataFrame(interaction_rows)
interactions.head()

,user_id,I01,I02,I03,I04,I05,I06,I07,I08
0,U01,0,1,0,1,1,1,1,0
1,U02,1,1,1,1,0,1,1,1
2,U03,1,1,1,1,1,1,1,1
3,U04,0,1,0,1,1,1,1,0
4,U05,0,1,1,1,0,1,0,0


## 5. Build user feature vectors

In [ ]:
feature_cols = [
    "casual", "elegant", "sporty",
    "neutral_colors", "bright_colors",
    "budget_sensitive"
]

X = users[feature_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X.head()

,casual,elegant,sporty,neutral_colors,bright_colors,budget_sensitive
0,4,2,4,2,5,4
1,5,5,1,4,1,1
2,3,4,4,5,5,4
3,5,1,2,2,5,2
4,5,1,2,2,1,1


## 6. Fit the KNN model

In [ ]:
knn = NearestNeighbors(n_neighbors=4, metric="euclidean")
knn.fit(X_scaled)

NearestNeighbors(metric='euclidean', n_neighbors=4)

## 7. Find similar users

In [ ]:
target_user_id = "U01"
target_idx = users.index[users["user_id"] == target_user_id][0]

distances, indices = knn.kneighbors([X_scaled[target_idx]])

pd.DataFrame({
    "neighbor_user_id": users.iloc[indices[0]]["user_id"].tolist(),
    "distance": distances[0].tolist()
})

,neighbor_user_id,distance
0,U01,0.000000
1,U17,1.801035
2,U04,2.398641
3,U14,2.482568


## 8. Recommend fashion items from similar users

In [ ]:
def recommend_items_for_user(target_user_id, top_n=3):
    target_idx = users.index[users["user_id"] == target_user_id][0]
    distances, indices = knn.kneighbors([X_scaled[target_idx]])

    neighbor_idxs = indices[0]
    neighbor_users = users.iloc[neighbor_idxs]["user_id"].tolist()

    target_row = interactions[interactions["user_id"] == target_user_id].iloc[0]
    neighbor_rows = interactions[interactions["user_id"].isin(neighbor_users)]

    item_scores = {}
    for item_id in items["item_id"]:
        if target_row[item_id] == 0:
            score = neighbor_rows[item_id].sum()
            if score > 0:
                item_scores[item_id] = score

    # fallback
    if len(item_scores) == 0:
        for item_id in items["item_id"]:
            item_scores[item_id] = neighbor_rows[item_id].sum()

    ranked = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    recs = pd.DataFrame(ranked, columns=["item_id", "neighbor_score"])
    recs = recs.merge(items[["item_id", "item_name"]], on="item_id", how="left")

    return recs

recommendations = recommend_items_for_user("U01", top_n=5)
recommendations

,item_id,neighbor_score,item_name
0,I03,1,Black Evening Dress


## 9. Explain the recommendation

In [ ]:
def explain_recommendation(target_user_id, item_id):
    target_idx = users.index[users["user_id"] == target_user_id][0]
    distances, indices = knn.kneighbors([X_scaled[target_idx]])
    neighbor_idxs = indices[0]

    target_profile = users.iloc[target_idx]
    neighbor_profiles = users.iloc[neighbor_idxs]
    neighbor_ids = neighbor_profiles["user_id"].tolist()

    contributing_neighbors = interactions[
        (interactions["user_id"].isin(neighbor_ids)) &
        (interactions[item_id] == 1)
    ]["user_id"].tolist()

    item_name = items.loc[items["item_id"] == item_id, "item_name"].iloc[0]

    shared_preferences = []
    if target_profile["casual"] >= 4:
        shared_preferences.append("casual style")
    if target_profile["elegant"] >= 4:
        shared_preferences.append("elegant style")
    if target_profile["sporty"] >= 4:
        shared_preferences.append("sporty style")
    if target_profile["neutral_colors"] >= 4:
        shared_preferences.append("neutral colors")
    if target_profile["bright_colors"] >= 4:
        shared_preferences.append("bright colors")

    if not shared_preferences:
        shared_preferences.append("similar overall style preferences")

    explanation = (
        f"Recommend '{item_name}' to {target_user_id} because "
        f"{len(contributing_neighbors)} similar users ({', '.join(contributing_neighbors)}) liked it. "
        f"Likely shared preferences: {', '.join(shared_preferences)}."
    )
    return explanation

for item_id in recommendations["item_id"].head(3):
    print(explain_recommendation("U01", item_id))
    print()

Recommend 'Black Evening Dress' to U01 because 1 similar users (U17) liked it. Likely shared preferences: casual style, sporty style, bright colors.



Why this model is interpretable

This recommender is interpretable because:
- recommendations come from similar users
- we can inspect the neighbors
- we can show the items they liked
- the explanation is based on real examples, not hidden rules

Limitations

KNN-based recommenders are simple and interpretable, but they also have limitations:
- they do not scale well to very large datasets
- recommendations depend strongly on the distance metric
- they do not explain which feature mattered most
- they may reinforce popularity bias or style bubbles

Optional exercises


Try extending this notebook:
1. change the number of neighbors k
2. switch from Euclidean distance to cosine distance
3. add a new feature such as preferred brands
4. build an item-based recommender instead of a user-based one
5. create a short text explanation for every recommendation

No submission required.